# Notebook 19 ? Complete Deep Learning Workflow

## Project: Predict retail returns

We use `data.csv` in this folder to predict whether a transaction line is a return. Each topic follows the same pattern: definition, example, reason, short code, and code explanation.

## 1. Problem definition

**Definition:** Binary classification predicts one of two labels.

**Example:** A transaction with negative quantity is labeled as a return.

**Why it is used?** The model learns to flag lines that may need review.

In [10]:
quantity = -2
is_return = int(quantity < 0)
print(is_return)

1


**Code Explanation:** `quantity < 0` creates the return label.

## 2. Dataset

**Definition:** A dataset is a table of examples and values.

**Example:** The local retail CSV has quantity, price, date, and country.

**Why it is used?** The project uses the requested file from this sprint folder.

In [12]:
from pathlib import Path
import pandas as pd
data_path = Path('data.csv')
if not data_path.exists():
    data_path = Path(r'C:\Users\hemak\Sprint 9 ? Deep Learning Fundamentals & Neural Networks\data.csv')
data = pd.read_csv(data_path, usecols=['Quantity', 'UnitPrice', 'InvoiceDate', 'Country'], encoding='latin-1')
data['InvoiceDate'] = pd.to_datetime(data['InvoiceDate'], format='mixed', dayfirst=True, errors='coerce')
print(data.shape)

(541909, 4)


**Code Explanation:** The code loads only needed columns and parses the mixed date formats.

## 3. Data inspection

**Definition:** Inspection checks rows, column types, and missing values.

**Example:** View the first rows and count missing entries.

**Why it is used?** We need to know what must be cleaned before training.

In [13]:
display(data.head())
print(data.dtypes)
print('Missing:', data.isna().sum().sum())

,Quantity,InvoiceDate,UnitPrice,Country
0,6,2010-01-12 08:26:00,2.55,United Kingdom
1,6,2010-01-12 08:26:00,3.39,United Kingdom
2,8,2010-01-12 08:26:00,2.75,United Kingdom
3,6,2010-01-12 08:26:00,3.39,United Kingdom
4,6,2010-01-12 08:26:00,3.39,United Kingdom


Quantity                int64
InvoiceDate    datetime64[us]
UnitPrice             float64
Country                   str
dtype: object
Missing: 0


**Code Explanation:** `head`, `dtypes`, and `isna` give a quick check of the table.

## 4. EDA

**Definition:** EDA summarizes data before model training.

**Example:** Count sales and returns to see the class balance.

**Why it is used?** A rare target class can make accuracy misleading.

In [14]:
data = data.dropna().copy()
data = data[data['UnitPrice'] > 0].copy()
data['is_return'] = (data['Quantity'] < 0).astype(int)
print(data['is_return'].value_counts())

is_return
0    530104
1      9288
Name: count, dtype: int64


**Code Explanation:** The target is 1 for a return and 0 for a sale.

## 5. Preprocessing

**Definition:** Preprocessing turns raw columns into model inputs.

**Example:** Use absolute quantity so its sign does not reveal the answer.

**Why it is used?** Clean and numeric input features make training easier.

In [15]:
from sklearn.model_selection import train_test_split
data['abs_quantity'] = data['Quantity'].abs()
data['hour'] = data['InvoiceDate'].dt.hour
data['weekday'] = data['InvoiceDate'].dt.dayofweek
features = ['abs_quantity', 'UnitPrice', 'hour', 'weekday']
X = data[features]
y = data['is_return']
if len(data) > 12000:
    X, _, y, _ = train_test_split(X, y, train_size=12000, stratify=y, random_state=19)

**Code Explanation:** We derive time values, remove the target clue, and sample rows for a quick run.

## 6. Train, validation, and test split

**Definition:** Separate data for learning, tuning, and final testing.

**Example:** Use a stratified split so each group has returns and sales.

**Why it is used?** Keep the test data untouched until the final evaluation.

In [16]:
from sklearn.model_selection import train_test_split
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, stratify=y, random_state=19)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=19)
print(len(X_train), len(X_val), len(X_test))

8400 1800 1800


**Code Explanation:** The data is split into 70% training, 15% validation, and 15% test.

## 7. Scale and prepare tensors

**Definition:** Scaling puts numeric features on similar ranges; tensors hold data for PyTorch.

**Example:** Fit the scaler on training rows, then transform all groups.

**Why it is used?** Fitting only on training rows avoids information leaking from the test set.

In [17]:
from sklearn.preprocessing import StandardScaler
import torch
from torch.utils.data import TensorDataset, DataLoader
scaler = StandardScaler().fit(X_train)
def make_loader(features, labels, shuffle=False):
    x = torch.tensor(scaler.transform(features), dtype=torch.float32)
    y_tensor = torch.tensor(labels.to_numpy(), dtype=torch.float32).view(-1, 1)
    return DataLoader(TensorDataset(x, y_tensor), batch_size=128, shuffle=shuffle)
train_loader = make_loader(X_train, y_train, shuffle=True)
val_x = torch.tensor(scaler.transform(X_val), dtype=torch.float32)
val_y = torch.tensor(y_val.to_numpy(), dtype=torch.float32).view(-1, 1)
test_x = torch.tensor(scaler.transform(X_test), dtype=torch.float32)
print(next(iter(train_loader))[0].shape)

torch.Size([128, 4])


**Code Explanation:** The loader returns small batches of features and labels.

## 8. Baseline model

**Definition:** A baseline is a simple result to compare with a neural network.

**Example:** Logistic regression is a quick binary classifier.

**Why it is used?** It shows whether a neural network improves on a simpler method.

In [18]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
baseline = LogisticRegression(class_weight='balanced', max_iter=500)
baseline.fit(scaler.transform(X_train), y_train)
baseline_pred = baseline.predict(scaler.transform(X_val))
print('Baseline F1:', round(f1_score(y_val, baseline_pred), 3))

Baseline F1: 0.03


**Code Explanation:** The baseline predicts validation rows and reports F1.

## 9. Neural network

**Definition:** A neural network learns patterns through connected layers.

**Example:** Use four input values, a hidden layer, and one output.

**Why it is used?** The hidden layer can learn relationships that are not straight lines.

In [19]:
from torch import nn
def build_model(hidden=16, dropout=0.0):
    return nn.Sequential(nn.Linear(len(features), hidden), nn.ReLU(), nn.Dropout(dropout), nn.Linear(hidden, 1))
model = build_model()
print(model)

Sequential(
  (0): Linear(in_features=4, out_features=16, bias=True)
  (1): ReLU()
  (2): Dropout(p=0.0, inplace=False)
  (3): Linear(in_features=16, out_features=1, bias=True)
)


**Code Explanation:** `Linear` layers learn weights; ReLU adds a non-linear step.

## 10. Loss function and optimizer

**Definition:** Loss measures errors; the optimizer updates weights.

**Example:** Binary cross-entropy works for two labels.

**Why it is used?** The optimizer uses loss gradients to improve the model.

In [20]:
positive_weight = torch.tensor([(y_train == 0).sum() / max((y_train == 1).sum(), 1)], dtype=torch.float32)
loss_fn = nn.BCEWithLogitsLoss(pos_weight=positive_weight)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
print(loss_fn)

BCEWithLogitsLoss()


**Code Explanation:** `pos_weight` gives the less common return class more influence.

## 11. Training and validation

**Definition:** Training updates weights; validation checks progress without updates.

**Example:** For one batch, calculate loss, backpropagate, and update weights.

**Why it is used?** Validation helps us check performance on data not used for updates.

In [21]:
for batch_x, batch_y in train_loader:
    optimizer.zero_grad()
    loss = loss_fn(model(batch_x), batch_y)
    loss.backward()
    optimizer.step()
    break
model.eval()
with torch.no_grad():
    print('Validation output:', model(val_x[:2]).flatten())

Validation output: tensor([0.1691, 0.1840])


**Code Explanation:** `backward` calculates gradients; `step` updates weights.

## 12. Hyperparameter experiments

**Definition:** Hyperparameters are settings chosen before training.

**Example:** Compare a 16-unit network with a 32-unit network using dropout.

**Why it is used?** Validation F1 helps us decide which configuration is better.

In [22]:
def train_model(candidate, epochs=4):
    optimizer = torch.optim.Adam(candidate.parameters(), lr=0.001)
    for _ in range(epochs):
        candidate.train()
        for batch_x, batch_y in train_loader:
            optimizer.zero_grad()
            loss_fn(candidate(batch_x), batch_y).backward()
            optimizer.step()
    return candidate
def probabilities(candidate, values):
    candidate.eval()
    with torch.no_grad():
        return torch.sigmoid(candidate(values)).numpy().ravel()
model_a = train_model(build_model(16))
model_b = train_model(build_model(32, 0.2))
f1_a = f1_score(y_val, probabilities(model_a, val_x) >= 0.5)
f1_b = f1_score(y_val, probabilities(model_b, val_x) >= 0.5)
print('16 units:', round(f1_a, 3), '| 32 + dropout:', round(f1_b, 3))

16 units: 0.017 | 32 + dropout: 0.036


**Code Explanation:** Both candidates use the same data split and training settings.

## 13. Error analysis

**Definition:** Error analysis finds predictions that do not match labels.

**Example:** A false negative is a return predicted as a sale.

**Why it is used?** The confusion matrix shows which mistakes happen.

In [23]:
from sklearn.metrics import confusion_matrix
best_model = model_b if f1_b >= f1_a else model_a
val_pred = (probabilities(best_model, val_x) >= 0.5).astype(int)
print(confusion_matrix(y_val, val_pred))

[[930 839]
 [ 15  16]]


**Code Explanation:** Rows are actual labels; columns are predicted labels.

## 14. Model improvement

**Definition:** Improvement means a change gives better validation results.

**Example:** The second network has more units and dropout.

**Why it is used?** Compare scores instead of assuming a larger network is better.

In [24]:
print('Basic F1:', round(f1_a, 3))
print('Improved F1:', round(f1_b, 3))

Basic F1: 0.017
Improved F1: 0.036


**Code Explanation:** Keep the improved network only if validation supports it.

## 15. Model selection

**Definition:** Choose a model using validation data, not test data.

**Example:** Select whichever candidate has higher validation F1.

**Why it is used?** This keeps the final test score fair.

In [25]:
selected_model = model_b if f1_b >= f1_a else model_a
print('Selected:', '32 units + dropout' if f1_b >= f1_a else '16 units')

Selected: 32 units + dropout


**Code Explanation:** The test set has not been used for this choice.

## 16. Final evaluation

**Definition:** Final evaluation measures the selected model on untouched test data.

**Example:** Report precision, recall, and F1 for returns.

**Why it is used?** Different metrics show different kinds of errors.

In [26]:
from sklearn.metrics import classification_report
test_pred = (probabilities(selected_model, test_x) >= 0.5).astype(int)
print(classification_report(y_test, test_pred, zero_division=0))

              precision    recall  f1-score   support

           0       0.98      0.51      0.67      1769
           1       0.02      0.52      0.04        31

    accuracy                           0.51      1800
   macro avg       0.50      0.51      0.35      1800
weighted avg       0.97      0.51      0.66      1800



**Code Explanation:** This is the final test score after model selection.

## 17. Save the model

**Definition:** Saving stores learned weights for future use.

**Example:** Write the selected weights to a file.

**Why it is used?** Saved weights avoid training from scratch again.

In [27]:
torch.save(selected_model.state_dict(), 'return_model.pt')
print('Saved return_model.pt')

Saved return_model.pt


**Code Explanation:** The file contains the selected network weights.

## 18. Inference

**Definition:** Inference uses a trained model on new data.

**Example:** Predict a return probability for one held-out row.

**Why it is used?** This is how the model is used after training.

In [28]:
selected_model.eval()
with torch.no_grad():
    probability = torch.sigmoid(selected_model(test_x[:1])).item()
print('Return probability:', round(probability, 3))

Return probability: 0.469


**Code Explanation:** A value near 1 means the model considers a return more likely.

## Limitation

This is a learning example. The random split may put similar transactions in multiple groups, and the features do not include product history. A later-time test and more useful product features would be better for real use.